<a href="https://colab.research.google.com/github/anshugupta27/Enhancing-LLMs-with-RAG-and-Efficient-Fine-Tuning-Techniques/blob/main/Anshu_Enhancing_LLMs_with_RAG_and_Efficient_Fine_Tuning_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 60.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [3]:
def load_documents(filepath='docs.txt'):
  with open(filepath, "r", encoding="utf-8") as f:
    return [line.strip() for line in f if line.strip()]

In [4]:
def embed_documents(documents):
  model = SentenceTransformer("all-MiniLM-L6-v2")
  embeddings = model.encode(documents)
  return embeddings, model

In [5]:
def build_faiss_index(embeddings):
  dim = embeddings.shape[1]
  index = faiss.IndexFlatL2(dim)
  index.add(np.array(embeddings))
  return index

In [6]:
def search_faiss(query, model, index, documents, top_k=3):
  query_embedding = model.encode([query])
  distance, indices = index.search(np.array(query_embedding), top_k)
  return [documents[i] for i in indices[0]]

In [7]:
docs = load_documents()
embeddings, model = embed_documents(docs)
index = build_faiss_index(embeddings)

query = "What is FAISS?"
results = search_faiss(query, model, index, docs, top_k=2)

print("top relevant documents:")
for r in results:
  print("-", r)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

top relevant documents:
- FAISS is a library for fast vector similarity search.
- ﻿Python is a popular programming language.


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
def format_prompt(query, context_docs):
  context = "\n".join(context_docs)
  return f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"


In [9]:
def generate_answer(query, retriever_fn, model, tokenizer, max_new_tokens=100):
  context_docs = retriever_fn(query)
  prompt = format_prompt(query, context_docs)
  inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

  with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
documents = load_documents("docs.txt")
embeddings, embed_model = embed_documents(documents)
faiss_index = build_faiss_index(embeddings)
model_name = "tiiuae/falcon-rw-1b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def retriever(query, top_k=3):
  return search_faiss(query, embed_model, faiss_index, documents, top_k=top_k)

if __name__ == "__main__":
  while True:
    query = input("Ask a question or type 'exit' to quit")
    if query.lower() == "exit":
      break
    answer = generate_answer(query, retriever, model, tokenizer)
    print("\n Answer:\n", answer)

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [ ]:
pip install peft accelerate bitsandbytes

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

lora_model_path = "timdettmers/guanaco-7b"  # or another real LoRA model

# Load LoRA config
peft_config = PeftConfig.from_pretrained(lora_model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    peft_config.base_model_name_or_path,
    load_in_8bit=True,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, lora_model_path)
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)
model.eval()


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
documents = load_documents("docs.txt")
embeddings, embed_model = embed_documents(documents)
faiss_index = build_faiss_index(embeddings)
model_name = "tiiuae/falcon-rw-1b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def retriever(query, top_k=3):
  return search_faiss(query, embed_model, faiss_index, documents, top_k=top_k)

if __name__ == "__main__":
  while True:
    query = input("Ask a question or type 'exit' to quit")
    if query.lower() == "exit":
      break
    answer = generate_answer(query, retriever, model, tokenizer)
    print("\n Answer:\n", answer)